In [0]:
# ============================================================
# Parameters - supplied by the Lakeflow Job via base_parameters
# ============================================================
# These widgets are populated automatically when run as a job task.
# To run interactively, set them manually via:
#   dbutils.widgets.text("catalog", "agents")
#   dbutils.widgets.text("schema", "mlops_demo")
#   etc.
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_name = dbutils.widgets.get("table_name")
n_samples = int(dbutils.widgets.get("n_samples"))
n_features = int(dbutils.widgets.get("n_features"))

full_table_name = f"{catalog}.{schema}.{table_name}"

# ============================================================
# Ensure catalog and schema exist
# ============================================================
# IMPORTANT: On workspaces with "Default Storage" enabled,
# CREATE CATALOG IF NOT EXISTS fails even for catalogs that
# already exist. The storage-root validation runs BEFORE the
# existence check, so the command errors out regardless.
#
# Workaround: Try USE CATALOG first (succeeds for any existing
# catalog without triggering storage validation). Only attempt
# CREATE CATALOG if USE CATALOG fails — and surface a clear
# error message if creation also fails due to storage config.
try:
    spark.sql(f"USE CATALOG `{catalog}`")
except Exception:
    # Catalog doesn't exist yet — attempt to create it
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")
        spark.sql(f"USE CATALOG `{catalog}`")
    except Exception as e:
        raise RuntimeError(
            f"Cannot access or create catalog '{catalog}'. "
            f"If your workspace uses Default Storage (no metastore storage root), "
            f"create the catalog via the Databricks UI or provide a MANAGED LOCATION. "
            f"Original error: {e}"
        ) from e

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

print(f"Target table: {full_table_name}")
print(f"Generating {n_samples} samples with {n_features} features")

In [0]:
from sklearn.datasets import make_classification
import pandas as pd

# Generate synthetic classification data using parameterized values
X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=max(2, n_features // 5),
    random_state=42
)

# Convert to pandas DataFrame
feature_cols = [f"feature_{i}" for i in range(n_features)]
pdf = pd.DataFrame(X, columns=feature_cols)
pdf["label"] = y

# Convert to Spark DataFrame
df = spark.createDataFrame(pdf)
display(df.limit(5))

In [0]:
import json

# Write to Delta table
df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)
row_count = spark.table(full_table_name).count()
print(f"Successfully saved {row_count} rows to {full_table_name}")

# Set task values for downstream tasks (train_model, batch_inference)
dbutils.jobs.taskValues.set(key="full_table_name", value=full_table_name)
dbutils.jobs.taskValues.set(key="row_count", value=row_count)
dbutils.jobs.taskValues.set(key="n_features", value=n_features)

print(f"\nTask values set:")
print(f"  full_table_name = {full_table_name}")
print(f"  row_count = {row_count}")
print(f"  n_features = {n_features}")

# Exit with structured output for the orchestrator
dbutils.notebook.exit(json.dumps({
    "status": "success",
    "table": full_table_name,
    "rows": row_count,
    "n_features": n_features
}))